## FarU project

### Concept demonstration:

Fine-tune a causal LLM (Mistral or other medical LLM) using LoRA to classify ANY suspicious focal pulmonary findings (lung nodules, masses, lesions, cysts, tumors, opacities, etc.).

Task:
Input: clinical sentence or paragraph
Output: 1 or 0 class

We develope code with the support of the AI co-pilot

In [1]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import Dataset
from datasets import load_dataset


In [2]:
# base_model = "mistralai/Mistral-Small-24B-Instruct-2501"
MODEL_NAME = 'mistralai/Mistral-Small-2409'

def get_model(model_name=MODEL_NAME):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,            # Yes/No output
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
        device_map="auto"
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="SEQ_CLS"
    )

    model = get_peft_model(model, lora_config)
    
    return model, tokenizer

In [ ]:

# def load_demo_dataset():

#     samples = [
#         {"text": "CT shows a 7mm pulmonary nodule.", "label": 1},
#         {"text": "Lungs clear, no suspicious lesions.", "label": 0},
#         {"text": "Mass-like growth in upper lobe.", "label": 1},
#         {"text": "No nodules or cystic abnormalities.", "label": 0},
#     ]

#     dataset = Dataset.from_list(samples)

#     return dataset

def load_synthetic_dataset():
    dataset = load_dataset("csv", data_files="data/synthetic_lung_findings.csv")
    dataset = dataset["train"].train_test_split(test_size=0.2)
    return dataset


In [4]:
model, tokenizer = get_model()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-Small-2409 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def preprocess(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    enc["label"] = batch["label"]
    return enc



In [ ]:
OUTPUT_DIR_BIN = "./mistral_lora_binary_classifier"
OUTPUT_DIR = "./mistral_lora_lung_change_classifier"
EPOCHS = 3
BATCH = 4
LR = 2e-4


def main():

    # dataset = load_demo_dataset()
    dataset = load_synthetic_dataset()['train']
    dataset = dataset.map(preprocess)

    dataset.set_format(type='torch')

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR_BIN,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        eval_strategy="no",   
        logging_strategy="steps",
        logging_steps=20,
        save_strategy="steps",
        save_steps=200,
        bf16=True,
        report_to="none"
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer
    )

    trainer.train()

    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)


In [9]:
main()

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

/tmp/ipykernel_2868249/2990623480.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
20,0.621500
40,0.000000
60,0.000000
80,0.000000
100,0.000000
120,0.000000
140,0.000000


In [ ]:
# Inference function

label_map = {0: "No", 1: "Yes"}  # suspicious lung change

def classify_lung_findings(text: str):
    """Return Yes/No for suspicious pulmonary abnormality."""
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding="max_length"
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    probs = F.softmax(logits, dim=-1)[0]
    pred = torch.argmax(probs).item()

    return {
        "prediction": label_map[pred],
        "prob_yes": float(probs[1]),
        "prob_no": float(probs[0])
    }

# -------------------------------------------------------
# 5. Example
# -------------------------------------------------------

examples = [
    'CT shows a 7 mm soft-tissue nodule in the right upper lobe.',
    'CT shows a small hyperintensity in the left upper lobe.',
    'Lungs are clear without any nodules or masses.',
    'No nodules, or nodules with benign features like complete central or popcorn calcification, or fat-containing nodules.',
    'Solid nodules less than 6mm, or ground-glass nodules less than 20mm.'
    'Solid nodules measuring 6-8mm, or ground-glass nodules greater than 20mm.',
    'A solid nodule between 8 and 15mm, or a part-solid nodule with a solid component of 6-8mm.',
    'A part-solid nodule greater than 10mm with a solid component greater than 5mm.',
    'A solid nodule greater than 10mm with worrisome features like lobulated or spiculated edges, no ground-glass border, and no inflammatory features.',
    'A nodule demonstrating signs of malignancy on CT, such as invasion of the chest wall or mediastinum.'
]

# Simple positive case
result = classify_lung_findings(examples[0])
print(result)

# Simple negative case
result = classify_lung_findings(examples[2])
print(result)

# Models must be intensively tuned to provide correct results do negative statements like "No nodules" correctly. See examples[3].
result = classify_lung_findings(examples[3])
print(result)



{'prediction': 'Yes', 'prob_yes': 1.0, 'prob_no': 0.0}
{'prediction': 'No', 'prob_yes': 0.0, 'prob_no': 1.0}
{'prediction': 'Yes', 'prob_yes': 1.0, 'prob_no': 0.0}
